In [ ]:
# /kaggle/input/objective-quest-2025/sample_submission.csv
# /kaggle/input/objective-quest-2025/train.csv
# /kaggle/input/objective-quest-2025/test.csv
# /kaggle/input/objective-quest-2025/file_putusan/file_putusan/doc_19683.txt

In [ ]:
pip install nltk


In [ ]:
pip install xgboost


In [ ]:
pip install lightgbm


# farrel notebook

In [20]:
# =============================================================================
#
# Prediksi Lama Hukuman Penjara - Solusi State-of-the-Art (V2)
# Kompetisi Kaggle NLP - Analisis Dokumen Hukum Indonesia
#
# Arsitektur:
# 1. Fine-Tuning Transformer Monolingual Indonesia (IndoBERT)
# 2. Arsitektur untuk Sekuens Panjang (Hingga 2048 token)
# 3. Validasi Silang K-Fold Bertingkat sebagai Inti Alur Kerja
# 4. Rekayasa Fitur Hukum yang Diperkaya
# 5. Kepala Model Neural Network dengan Attention Pooling
# 6. Fungsi Kerugian Huber Loss yang Robust
# 7. Stacking Ensemble dengan Model Meta
#
# =============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import os
from pathlib import Path
import gc
import joblib
from collections import Counter
import pickle
from tqdm.auto import tqdm

# Deep Learning & Transformers
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

# Traditional ML for Stacking Ensemble
from sklearn.linear_model import RidgeCV
import lightgbm as lgb

# NLP Libraries
try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
except ImportError:
    print("Sastrawi not found. Installing...")
    os.system('pip install Sastrawi')
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

import unicodedata

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8')



# 1. KONFIGURASI UTAMA

In [34]:
# =============================================================================
# 1. KONFIGURASI UTAMA
# =============================================================================
class CFG:
    """Konfigurasi terpusat untuk eksperimen."""
    # --- Model & Tokenizer ---
    MODEL_NAME = "indobenchmark/indobert-base-p1"
    MAX_LENGTH = 2048  # Peningkatan signifikan dari 512
    
    # --- Pelatihan ---
    N_FOLDS = 5
    EPOCHS = 5
    EARLY_STOPPING_PATIENCE = 2
    BATCH_SIZE = 16
    ACCUMULATION_STEPS = 1
    LEARNING_RATE = 2e-5
    WEIGHT_DECAY = 0.01
    LOSS_FUNCTION = "huber"  # Opsi: "mse", "huber"
    HUBER_DELTA = 1.0
    
    # --- Prapemrosesan ---
    USE_STEMMING = False  # Stemming sebagai hyperparameter
    
    # --- Umum ---
    SEED = 42
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
def set_seed(seed):
    """Menetapkan random seed untuk reproduktifitas."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG.SEED)
print(f"Using device: {CFG.DEVICE}")




Using device: cuda


# 2. PRAPEMROSESAN & REKAYASA FITUR TINGKAT LANJUT

In [22]:
# =============================================================================
# 2. PRAPEMROSESAN & REKAYASA FITUR TINGKAT LANJUT
# =============================================================================
class AdvancedIndonesianLegalPreprocessor:
    """Prapemrosesan yang diperkaya untuk dokumen hukum Indonesia."""
    
    def __init__(self, use_stemming=False):
        self.stop_words = self._load_indonesian_stopwords()
        self.legal_patterns = self._compile_advanced_legal_patterns()
        self.use_stemming = use_stemming
        if self.use_stemming:
            factory = StemmerFactory()
            self.stemmer = factory.create_stemmer()

    def _load_indonesian_stopwords(self):
        """Daftar stopword yang lebih komprehensif."""
        # Sumber: https://github.com/stopwords-iso/stopwords-id
        base_stopwords = ['ada', 'adalah', 'adanya', 'adapun', 'agak', 'agaknya', 'agar', 'akan', 'akankah', 'akhir', 'akhiri', 'akhirnya', 'aku', 'akulah', 'amat', 'amatlah', 'anda', 'andalah', 'antar', 'antara', 'antaranya', 'apa', 'apaan', 'apabila', 'apakah', 'apalagi', 'apatah', 'artinya', 'asal', 'asalkan', 'atas', 'atau', 'ataukah', 'ataupun', 'awal', 'awalnya', 'bagai', 'bagaikan', 'bagaimana', 'bagaimanakah', 'bagaimanapun', 'bagi', 'bagian', 'bahkan', 'bahwa', 'bahwasanya', 'baik', 'bakal', 'bakalan', 'balik', 'banyak', 'bapak', 'baru', 'bawah', 'beberapa', 'begini', 'beginian', 'beginikah', 'beginilah', 'begitu', 'begitukah', 'begitulah', 'begitupun', 'bekerja', 'belakang', 'belakangan', 'belum', 'belumlah', 'benar', 'benarkah', 'benarlah', 'berada', 'berakhir', 'berakhirlah', 'berakhirnya', 'berapa', 'berapakah', 'berapalah', 'berapapun', 'berarti', 'berawal', 'berbagai', 'berdatangan', 'beri', 'berikan', 'berikut', 'berikutnya', 'berjumlah', 'berkali-kali', 'berkata', 'berkehendak', 'berkeinginan', 'berkenaan', 'berlainan', 'berlalu', 'berlangsung', 'berlebihan', 'bermacam', 'bermacam-macam', 'bermaksud', 'bermula', 'bersama', 'bersama-sama', 'bersiap', 'bersiap-siap', 'bertanya', 'bertanya-tanya', 'berturut', 'berturut-turut', 'bertutur', 'berujar', 'berupa', 'besar', 'betul', 'betulkah', 'biasa', 'biasanya', 'bila', 'bilakah', 'bisa', 'bisakah', 'boleh', 'bolehkah', 'bolehlah', 'buat', 'bukan', 'bukankah', 'bukanlah', 'bukannya', 'bulan', 'bung', 'cara', 'caranya', 'cukup', 'cukupkah', 'cukuplah', 'cuma', 'dahulu', 'dalam', 'dan', 'dapat', 'dari', 'daripada', 'datang', 'demi', 'demikian', 'demikianlah', 'dengan', 'depan', 'di', 'dia', 'diakhiri', 'diakhirinya', 'dialah', 'diantara', 'diantaranya', 'diberi', 'diberikan', 'diberikannya', 'dibuat', 'dibuatnya', 'didapat', 'didatangkan', 'digunakan', 'diibaratkan', 'diibaratkannya', 'diingat', 'diingatkan', 'diinginkan', 'dijawab', 'dijelaskan', 'dijelaskannya', 'dikarenakan', 'dikatakan', 'dikatakannya', 'dikerjakan', 'diketahui', 'diketahuinya', 'dikiranya', 'dilakukan', 'dilalui', 'dilihat', 'dilihatnya', 'dimaksud', 'dimaksudkan', 'dimaksudkannya', 'dimaksudnya', 'diminta', 'dimintai', 'dimisalkan', 'dimulai', 'dimulailah', 'dimulainya', 'dimungkinkan', 'dini', 'dipastikan', 'diperbuat', 'diperbuatnya', 'dipergunakan', 'diperkirakan', 'diperlihatkan', 'diperlukan', 'diperlukannya', 'dipersoalkan', 'dipertanyakan', 'dipunyai', 'diri', 'dirinya', 'disampaikan', 'disebut', 'disebutkan', 'disebutkannya', 'disini', 'disinilah', 'ditambahkan', 'ditambahkannya', 'ditanya', 'ditanyai', 'ditanyakan', 'ditegaskan', 'ditujukan', 'ditunjuk', 'ditunjuki', 'ditunjukkan', 'ditunjukkannya', 'dituntun', 'diturunkan', 'dituturkan', 'dituturkannya', 'diucapkan', 'diucapkannya', 'diungkapkan', 'dong', 'dua', 'dulu', 'empat', 'enggak', 'enggaknya', 'entah', 'entahlah', 'guna', 'gunakan', 'hal', 'hampir', 'hanya', 'hanyalah', 'hari', 'harus', 'haruslah', 'harusnya', 'hendak', 'hendaklah', 'hendaknya', 'hingga', 'ia', 'ialah', 'ibarat', 'ibaratnya', 'ibu', 'ikut', 'ingat', 'ingat-ingat', 'ingin', 'inginkah', 'inginkan', 'ini', 'inikah', 'inilah', 'itu', 'itukah', 'itulah', 'jadi', 'jadilah', 'jadinya', 'jangan', 'jangankan', 'janganlah', 'jauh', 'jawab', 'jawaban', 'jawabnya', 'jelas', 'jelaskan', 'jelaslah', 'jelasnya', 'jika', 'jikalau', 'juga', 'jumlah', 'jumlahnya', 'justru', 'kala', 'kalau', 'kalaulah', 'kalaupun', 'kali', 'kalian', 'kami', 'kamilah', 'kamu', 'kamulah', 'kan', 'kapan', 'kapankah', 'kapanpun', 'karena', 'karenanya', 'kasus', 'kata', 'katakan', 'katakanlah', 'katanya', 'ke', 'keadaan', 'kebetulan', 'kecil', 'kedua', 'keduanya', 'keinginan', 'kelak', 'kelima', 'keluar', 'kembali', 'kemudian', 'kemungkinan', 'kemungkinannya', 'kenapa', 'kepada', 'kepadanya', 'kesampaian', 'keseluruhan', 'keseluruhannya', 'keterlaluan', 'ketika', 'khususnya', 'kini', 'kinilah', 'kira', 'kira-kira', 'kiranya', 'kita', 'kitalah', 'kok', 'kurang', 'lagi', 'lagian', 'lah', 'lain', 'lainnya', 'lalu', 'lama', 'lamanya', 'lanjut', 'lanjutnya', 'lebih', 'lewat', 'lima', 'luar', 'macam', 'maka', 'makanya', 'makin', 'malah', 'malahan', 'mampu', 'mampukah', 'mana', 'manakala', 'manalagi', 'masa', 'masalah', 'masalahnya', 'masih', 'masihkah', 'masing', 'masing-masing', 'mau', 'maupun', 'melainkan', 'melakukan', 'melalui', 'melihat', 'melihatnya', 'memang', 'memastikan', 'memberi', 'memberikan', 'membuat', 'memerlukan', 'memintakan', 'memisalkan', 'memperbuat', 'mempergunakan', 'memperkirakan', 'memperlihatkan', 'mempersiapkan', 'mempersoalkan', 'mempertanyakan', 'mempunyai', 'memulai', 'memungkinkan', 'menaiki', 'menambahkan', 'menandaskan', 'menanti', 'menanti-nanti', 'menantikan', 'menanya', 'menanyai', 'menanyakan', 'mendapat', 'mendapatkan', 'mendatang', 'mendatangi', 'mendatangkan', 'menegaskan', 'mengakhiri', 'mengapa', 'mengatakan', 'mengatakannya', 'mengenai', 'mengerjakan', 'mengetahui', 'menggunakan', 'menghendaki', 'mengibaratkan', 'mengibaratkannya', 'mengingat', 'mengingatkan', 'menginginkan', 'mengira', 'mengucapkan', 'mengucapkannya', 'mengungkapkan', 'menjadi', 'menjawab', 'menjelaskan', 'menuju', 'menunjuk', 'menunjuki', 'menunjukkan', 'menurut', 'menuturkan', 'menyampaikan', 'menyangkut', 'menyatakan', 'menyebutkan', 'menyeluruh', 'menyiapkan', 'merasa', 'mereka', 'merekalah', 'merupakan', 'meski', 'meskipun', 'meyakini', 'meyakinkan', 'minta', 'mirip', 'misal', 'misalkan', 'misalnya', 'mula', 'mulai', 'mulailah', 'mulanya', 'mungkin', 'mungkinkah', 'nah', 'naik', 'namun', 'nanti', 'nantinya', 'nyaris', 'nyatanya', 'oleh', 'olehnya', 'pada', 'padahal', 'padanya', 'pak', 'paling', 'panjang', 'pantas', 'para', 'pasti', 'pastilah', 'penting', 'pentingnya', 'per', 'percuma', 'perlu', 'perlukah', 'perlunya', 'pernah', 'persoalan', 'pertama', 'pertama-tama', 'pertanyaan', 'pertanyakan', 'pihak', 'pihaknya', 'pukul', 'pula', 'pun', 'punya', 'rasa', 'rasanya', 'rata', 'rupanya', 'saat', 'saatnya', 'saja', 'sajalah', 'saling', 'sama', 'sama-sama', 'sambil', 'sampai', 'sampai-sampai', 'sampaikan', 'sana', 'sangat', 'sangatlah', 'satu', 'saya', 'sayalah', 'se', 'sebab', 'sebabnya', 'sebagai', 'sebagaimana', 'sebagainya', 'sebagian', 'sebaik', 'sebaik-baiknya', 'sebaiknya', 'sebaliknya', 'sebanyak', 'sebelum', 'sebelumnya', 'sebenarnya', 'seberapa', 'sebesar', 'sebetulnya', 'sebisanya', 'sebuah', 'sebut', 'sebutlah', 'sebutnya', 'secara', 'secukupnya', 'sedang', 'sedangkan', 'sedemikian', 'sedikit', 'sedikitnya', 'seenaknya', 'segala', 'segalanya', 'segera', 'seharusnya', 'sehingga', 'seingat', 'sejak', 'sejauh', 'sejenak', 'sejumlah', 'sekadar', 'sekadarnya', 'sekali', 'sekali-kali', 'sekalian', 'sekaligus', 'sekalipun', 'sekarang', 'sekaranglah', 'sekecil', 'seketika', 'sekiranya', 'sekitar', 'sekitarnya', 'sekurang-kurangnya', 'sekurangnya', 'sela', 'selain', 'selaku', 'selalu', 'selama', 'selama-lamanya', 'selamanya', 'selanjutnya', 'seluruh', 'seluruhnya', 'semacam', 'semakin', 'semampu', 'semampunya', 'semasa', 'semasih', 'semata', 'semata-mata', 'semaunya', 'sementara', 'semisal', 'semisalnya', 'sempat', 'semua', 'semuanya', 'semula', 'sendiri', 'sendirian', 'sendirinya', 'seolah', 'seolah-olah', 'seorang', 'sepanjang', 'sepantasnya', 'sepantasnyalah', 'seperlunya', 'seperti', 'sepertinya', 'sepeserpun', 'sering', 'seringnya', 'serta', 'serupa', 'sesaat', 'sesampai', 'sesegera', 'sesekali', 'seseorang', 'sesuatu', 'sesuatunya', 'sesudah', 'sesudahnya', 'setelah', 'setempat', 'setengah', 'seterusnya', 'setiap', 'setiba', 'setibanya', 'setidak-tidaknya', 'setidaknya', 'setinggi', 'seusai', 'sewaktu', 'siap', 'siapa', 'siapakah', 'siapapun', 'sini', 'sinilah', 'suatu', 'sudah', 'sudahkah', 'sudahlah', 'supaya', 'tadi', 'tadinya', 'tahu', 'tahun', 'tak', 'tambah', 'tambahnya', 'tampak', 'tampaknya', 'tandas', 'tandasnya', 'tanpa', 'tanya', 'tanyakan', 'tanyanya', 'tapi', 'tegas', 'tegasnya', 'telah', 'tempat', 'tengah', 'tentang', 'tentu', 'tentulah', 'tentunya', 'tepat', 'terakhir', 'terasa', 'terbanyak', 'terdahulu', 'terdapat', 'terdiri', 'terhadap', 'terhadapnya', 'teringat', 'teringat-ingat', 'terjadi', 'terjadilah', 'terjadinya', 'terkira', 'terlalu', 'terlebih', 'terlihat', 'termasuk', 'ternyata', 'tersampaikan', 'tersebut', 'tersebutlah', 'tertentu', 'tertuju', 'terus', 'terutama', 'tetap', 'tetapi', 'tiap', 'tiba', 'tiba-tiba', 'tidak', 'tidakkah', 'tidaklah', 'tiga', 'toh', 'tunjuk', 'turut', 'tutur', 'tuturnya', 'ucap', 'ucapnya', 'ujar', 'ujarnya', 'umum', 'umumnya', 'ungkap', 'ungkapnya', 'untuk', 'usah', 'usai', 'waduh', 'wah', 'wahai', 'waktu', 'waktunya', 'walau', 'walaupun', 'wong', 'yaitu', 'yakin', 'yakni', 'yang']
        legal_stopwords = ['menimbang', 'mengingat', 'memperhatikan', 'mengadili', 'menetapkan', 'menyatakan', 'membebankan', 'amar', 'putusan', 'demi', 'keadilan', 'berdasarkan', 'ketuhanan', 'maha', 'esa', 'terdakwa', 'penuntut', 'umum', 'majelis', 'hakim']
        return set(base_stopwords + legal_stopwords)

    def _compile_advanced_legal_patterns(self):
        """Pola regex yang sama seperti sebelumnya."""
        patterns = {
            'pasal': re.compile(r'pasal\s+(\d+(?:\s*[a-z])?(?:\s*ayat\s*\(\d+\))?)', re.IGNORECASE),
            'undang_undang': re.compile(r'undang[- ]undang\s+(?:republik\s+indonesia\s+)?(?:nomor\s+)?(\d+)\s*tahun\s*(\d{4})', re.IGNORECASE),
            'kuhp': re.compile(r'k\.?u\.?h\.?p\.?|kitab\s+undang[- ]undang\s+hukum\s+pidana', re.IGNORECASE),
            'pidana_penjara': re.compile(r'pidana\s+penjara\s+(?:selama\s+|paling\s+(?:lama|singkat)\s+)?(\d+)\s+(tahun|bulan|hari)', re.IGNORECASE),
            'denda': re.compile(r'(?:pidana\s+)?denda\s+(?:sebesar\s+)?(?:rp\.?\s*)?([\d,.]+)', re.IGNORECASE),
            'hukuman_mati': re.compile(r'pidana\s+mati|hukuman\s+mati|dihukum\s+mati', re.IGNORECASE),
            'seumur_hidup': re.compile(r'(?:pidana\s+)?penjara\s+seumur\s+hidup', re.IGNORECASE),
            'tindak_pidana': re.compile(r'tindak\s+pidana\s+([^,\n\.]+)', re.IGNORECASE),
            'dakwaan': re.compile(r'dakwaan\s+(?:primair|subsidair|alternatif|tunggal)', re.IGNORECASE),
            # Inside the patterns dictionary in _compile_advanced_legal_patterns
            'mengadili_spaced': re.compile(r'm\s*e\s*n\s*g\s*a\s*d\s*i\s*l\s*i', re.IGNORECASE),
            # Inside the patterns dictionary in _compile_advanced_legal_patterns
            'residivis': re.compile(r'residivis|pernah dihukum', re.IGNORECASE),
            'tulang_punggung': re.compile(r'tulang punggung keluarga', re.IGNORECASE),
            'menyesali': re.compile(r'menyesali perbuatannya|mengakui terus terang', re.IGNORECASE),
        }
        return patterns

    def extract_features(self, text):
        """Ekstraksi fitur yang diperkaya dengan fitur kepadatan dan posisional."""
        features = {}
        doc_len = len(text.split())
        if doc_len == 0:
            # Return a dictionary with all possible keys to ensure consistent columns
            base_features = {f'{p}_count': 0 for p in self.legal_patterns}
            base_features.update({f'{p}_density': 0.0 for p in self.legal_patterns})

            extra_features = {
                'seumur_hidup_in_amar': 0, 'hukuman_mati_in_amar': 0,
                'doc_length': 0, 'unique_word_ratio': 0.0,
                'pidana_penjara_in_mengadili': 0, 'denda_in_mengadili': 0
            }
            return {**base_features, **extra_features}
    
        # Use the cleaned text for feature extraction for consistency
        cleaned_text = self.clean_text(text)
        text_lower = cleaned_text.lower()
        
        # Ekstraksi berbasis pola & kepadatan
        for name, pattern in self.legal_patterns.items():
            count = len(pattern.findall(text_lower))
            features[f'{name}_count'] = count
            features[f'{name}_density'] = count / doc_len
    
        # --- NEW: Isolate and analyze the MENGADILI (verdict) section ---
        mengadili_text = ""
        # Find the verdict section using both normal and spaced patterns
        mengadili_match = self.legal_patterns['mengadili_spaced'].search(text.lower())
        if mengadili_match:
            # Take the text after "M E N G A D I L I"
            mengadili_text = text.lower()[mengadili_match.end():]
    
        features['pidana_penjara_in_mengadili'] = 0
        features['denda_in_mengadili'] = 0
        if mengadili_text:
            # Check for key sentencing terms ONLY within the verdict text
            features['pidana_penjara_in_mengadili'] = len(self.legal_patterns['pidana_penjara'].findall(mengadili_text))
            features['denda_in_mengadili'] = len(self.legal_patterns['denda'].findall(mengadili_text))
            
        # Fitur posisional (sangat prediktif)
        sections = re.split(r'\b(menimbang|mengingat|mengadili|amar putusan)\b', text.lower(), flags=re.IGNORECASE)
        in_amar_putusan = False
        features['seumur_hidup_in_amar'] = 0
        features['hukuman_mati_in_amar'] = 0
        for section in sections:
            if 'amar putusan' in section:
                in_amar_putusan = True
            if in_amar_putusan:
                features['seumur_hidup_in_amar'] += section.count('seumur hidup')
                features['hukuman_mati_in_amar'] += section.count('pidana mati')
    
        # Fitur lain
        features['doc_length'] = doc_len
        features['unique_word_ratio'] = len(set(text_lower.split())) / doc_len
        
        return features

    def clean_text(self, text):
        """Pembersihan teks dengan opsi stemming."""
        if pd.isna(text):
            return ""
        
        text = str(text)

        # --- NEW: Hapus header dan footer yang berulang ---
        # Hapus header
        text = re.sub(r'hkama\s*ahkamah Agung Repub\s*ahkamah Agung Republik Indonesia\s*mah Agung Republik Indonesia\s*blik Indonesi\s*Direktori Putusan Mahkamah Agung Republik Indonesia', ' ', text, flags=re.IGNORECASE)
        # Hapus footer disclaimer dan informasi halaman
        text = re.sub(r'Halaman \d+ dari \d+.*?\(ext\.318\)|transparansi dan akuntabilitas.*?\(ext\.318\)', ' ', text, flags=re.DOTALL | re.IGNORECASE)
        # --- End of new code ---
    
        text = unicodedata.normalize('NFKD', text)
        text = text.lower()
        
        # Hapus URL, email, dan karakter non-alfanumerik kecuali yang penting
        text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
        text = re.sub(r'\S+@\S+', '', text)
        text = re.sub(r'[^\w\s.,-]', ' ', text)

        # Inside clean_text, before the "Normalisasi spasi" section

        # Normalisasi angka seperti "1 (satu)" menjadi "1"
        text = re.sub(r'(\d+)\s*\([^)]*\)', r'\1', text)
        
        # Normalisasi format mata uang seperti "Rp.5.000,-" menjadi "5000"
        text = re.sub(r'rp\s?\.?\s?([\d,.]+),?-?', lambda m: m.group(1).replace('.', '').replace(',', ''), text, flags=re.IGNORECASE)
        
        # Normalisasi spasi
        text = re.sub(r'\s+', ' ', text).strip()
        
        # Stemming (opsional)
        if self.use_stemming:
            text = self.stemmer.stem(text)
            
        return text

# 3. DATASET & MODEL PYTORCH UNTUK FINE-TUNING

In [23]:
# =============================================================================
# 3. DATASET & MODEL PYTORCH UNTUK FINE-TUNING
# =============================================================================
class LegalFineTuningDataset(Dataset):
    """Dataset untuk fine-tuning, menangani tokenisasi on-the-fly."""
    def __init__(self, df, tokenizer, cfg, texts, legal_features):
        self.df = df
        self.texts = texts
        self.legal_features = legal_features
        self.tokenizer = tokenizer
        self.cfg = cfg
        self.targets = df['lama hukuman (bulan)'].values if 'lama hukuman (bulan)' in df.columns else None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.texts[idx]
        features = self.legal_features[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.cfg.MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'legal_features': torch.tensor(features, dtype=torch.float)
        }
        
        if self.targets is not None:
            target_val = self.targets[idx]
            # Transformasi Log1p
            if target_val > 0 and target_val!= 88888:
                target_val = np.log1p(target_val)
            item['targets'] = torch.tensor(target_val, dtype=torch.float)
            
        return item

class AttentionPooling(nn.Module):
    """Pooling dengan mekanisme atensi yang dapat dilatih."""
    def __init__(self, in_features):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.LayerNorm(in_features),
            nn.GELU(),
            nn.Linear(in_features, 1)
        )

    def forward(self, last_hidden_state, attention_mask):
        w = self.attention(last_hidden_state).float()
        w[attention_mask==0] = float('-inf')
        w = torch.softmax(w, 1)
        attention_pools = torch.sum(w * last_hidden_state, dim=1)
        return attention_pools

class LegalFineTuningModel(nn.Module):
    """Model fine-tuning yang mengintegrasikan Transformer dan fitur rekayasa."""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.config = AutoConfig.from_pretrained(cfg.MODEL_NAME, output_hidden_states=True)

        # --- FIX: Update the max position embeddings to match your MAX_LENGTH ---
        self.config.max_position_embeddings = cfg.MAX_LENGTH
        
        # --- FIX: Add ignore_mismatched_sizes=True ---
        self.transformer = AutoModel.from_pretrained(
            cfg.MODEL_NAME, 
            config=self.config, 
            ignore_mismatched_sizes=True
        )
        self.pool = AttentionPooling(self.config.hidden_size)
        
        # Placeholder untuk dimensi fitur hukum, akan di-update nanti
        self.legal_feature_dim = 0 
        
        self.head = nn.Sequential(
            nn.Linear(self.config.hidden_size + self.legal_feature_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def update_feature_dim(self, dim):
        """Memperbarui dimensi input kepala model setelah fitur rekayasa dibuat."""
        self.legal_feature_dim = dim
        self.head = nn.Sequential(
            nn.Linear(self.config.hidden_size + self.legal_feature_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )
        self.head.to(self.cfg.DEVICE)

    def forward(self, input_ids, attention_mask, legal_features):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state
        pooled_output = self.pool(last_hidden_state, attention_mask)
        
        combined_features = torch.cat([pooled_output, legal_features], dim=1)
        
        return self.head(combined_features)

# 4. ALUR KERJA PELATIHAN & INFERENSI

In [ ]:
# =============================================================================
# 4. ALUR KERJA PELATIHAN & INFERENSI
# =============================================================================

from torch.cuda.amp import GradScaler, autocast
import pickle

def get_loss_fn(cfg):
    if cfg.LOSS_FUNCTION == "huber":
        return nn.HuberLoss(delta=cfg.HUBER_DELTA)
    else: # default to mse
        return nn.MSELoss()

def train_fn(train_loader, model, criterion, optimizer, scheduler, device, scaler):
    model.train()
    total_loss = 0
    
    # 1. Enumerate the dataloader to get a step count
    for step, batch in enumerate(tqdm(train_loader, desc="Training")):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        legal_features = batch['legal_features'].to(device)
        targets = batch['targets'].to(device)
        
        with autocast():
            outputs = model(input_ids, attention_mask, legal_features)
            loss = criterion(outputs.squeeze(), targets)
            
            # 2. Normalize the loss to account for accumulation
            loss = loss / CFG.ACCUMULATION_STEPS
            
        # 3. Scale the normalized loss and backpropagate (this happens on every step)
        scaler.scale(loss).backward()
        
        # 4. Update weights and scheduler ONLY every ACCUMULATION_STEPS
        if (step + 1) % CFG.ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            scheduler.step()
        
        total_loss += loss.item() * CFG.ACCUMULATION_STEPS # Re-scale loss for logging
        
    return total_loss / len(train_loader)

def valid_fn(valid_loader, model, criterion, device):
    """Fungsi untuk validasi."""
    model.eval()
    total_loss = 0
    preds = []
    with torch.no_grad():
        for batch in tqdm(valid_loader, desc="Validation"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            legal_features = batch['legal_features'].to(device)
            targets = batch['targets'].to(device)
            
            outputs = model(input_ids, attention_mask, legal_features)
            loss = criterion(outputs.squeeze(), targets)
            total_loss += loss.item()
            preds.append(outputs.squeeze().cpu().numpy())
            
    return total_loss / len(valid_loader), np.concatenate(preds)

def run_fold_training(fold, train_df, tokenizer, cfg, texts, legal_features):
    """Menjalankan pelatihan dan validasi untuk satu fold."""
    print(f"\n========== FOLD {fold + 1} ==========")
    
    # --- Pembagian Data ---
    train_indices = train_df[train_df['fold']!= fold].index
    valid_indices = train_df[train_df['fold'] == fold].index
    
    train_fold_df = train_df.loc[train_indices].reset_index(drop=True)
    valid_fold_df = train_df.loc[valid_indices].reset_index(drop=True)
    
    train_texts = [texts[i] for i in train_indices]
    valid_texts = [texts[i] for i in valid_indices]
    
    train_legal_features = legal_features[train_indices]
    valid_legal_features = legal_features[valid_indices]

    # --- Dataset & DataLoader ---
    train_dataset = LegalFineTuningDataset(train_fold_df, tokenizer, cfg, train_texts, train_legal_features)
    valid_dataset = LegalFineTuningDataset(valid_fold_df, tokenizer, cfg, valid_texts, valid_legal_features)
    
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        drop_last=True  # <-- ADD THIS LINE
    )
    valid_loader = DataLoader(valid_dataset, batch_size=cfg.BATCH_SIZE * 2, shuffle=False, num_workers=4, pin_memory=True)
    
    # --- Model, Loss, Optimizer ---
    model = LegalFineTuningModel(cfg)
    model.update_feature_dim(train_legal_features.shape[1]) # Update dimensi fitur

    # Check if multiple GPUs are available and wrap the model
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
        
    model.to(cfg.DEVICE)
    
    criterion = get_loss_fn(cfg)
    optimizer = optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)
    
    num_training_steps = len(train_loader) * cfg.EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)
    
    # --- Loop Pelatihan ---
    best_loss = float('inf')
    patience_counter = 0 
    oof_preds = None

    scaler = GradScaler()
    
    for epoch in range(cfg.EPOCHS):
        train_loss = train_fn(train_loader, model, criterion, optimizer, scheduler, cfg.DEVICE, scaler)
        valid_loss, valid_preds = valid_fn(valid_loader, model, criterion, cfg.DEVICE)
        
        print(f"Epoch {epoch+1}/{cfg.EPOCHS} -> Train Loss: {train_loss:.4f}, Valid Loss: {valid_loss:.4f}")
        
        if valid_loss < best_loss:
            best_loss = valid_loss
            oof_preds = valid_preds
            # Save the underlying model's state dict
            if isinstance(model, nn.DataParallel):
                torch.save(model.module.state_dict(), f"best_model_fold_{fold}.pth")
            else:
                torch.save(model.state_dict(), f"best_model_fold_{fold}.pth")
            print("  -> Model saved!")
            patience_counter = 0
        else:
            patience_counter += 1 # <-- INCREMENT counter if no improvement
            
        # --- ADD THIS BLOCK for the early stopping check ---
        if patience_counter >= cfg.EARLY_STOPPING_PATIENCE:
            print(f"Early stopping triggered after {epoch + 1} epochs.")
            break # <-- EXIT the loop for this fold

    # Save the OOF predictions for this fold to a file
    oof_data = {'preds': oof_preds, 'indices': valid_indices}
    with open(f"oof_fold_{fold}.pkl", "wb") as f:
        pickle.dump(oof_data, f)
    print(f"OOF predictions for fold {fold} saved.")
    
    return oof_preds, valid_indices

def run_full_training(train_df, tokenizer, cfg, texts, legal_features):
    """Menjalankan pelatihan pada seluruh dataset training."""
    print("\\n========== TRAINING ON FULL DATASET ==========")
    
    # --- Dataset & DataLoader ---
    train_dataset = LegalFineTuningDataset(train_df, tokenizer, cfg, texts, legal_features)
    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        drop_last=True
    )
    
    # --- Model, Loss, Optimizer ---
    model = LegalFineTuningModel(cfg)
    model.update_feature_dim(legal_features.shape[1])

    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs!")
        model = nn.DataParallel(model)
        
    model.to(cfg.DEVICE)
    
    criterion = get_loss_fn(cfg)
    optimizer = optim.AdamW(model.parameters(), lr=cfg.LEARNING_RATE, weight_decay=cfg.WEIGHT_DECAY)
    
    num_training_steps = len(train_loader) * cfg.EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)
    
    scaler = GradScaler()
    
    # --- Loop Pelatihan ---
    for epoch in range(cfg.EPOCHS):
        train_loss = train_fn(train_loader, model, criterion, optimizer, scheduler, cfg.DEVICE, scaler)
        print(f"Epoch {epoch+1}/{cfg.EPOCHS} -> Train Loss: {train_loss:.4f}")
        
    # --- Simpan Model Final ---
    model_path = "final_model.pth"
    if isinstance(model, nn.DataParallel):
        torch.save(model.module.state_dict(), model_path)
    else:
        torch.save(model.state_dict(), model_path)
    print(f"Model final disimpan di '{model_path}'")

# 5. ALUR KERJA UTAMA

## 1. Muat Data

In [25]:
# 1. Muat Data
print("1. Memuat data...")
train_df = pd.read_csv("/kaggle/input/objective-quest-2025/train.csv")
test_df = pd.read_csv("/kaggle/input/objective-quest-2025/test.csv")

1. Memuat data...


## 2. Prapemrosesan & Rekayasa Fitur

In [27]:
# 2. Prapemrosesan & Rekayasa Fitur
print("\n2. Prapemrosesan teks dan rekayasa fitur...")
preprocessor = AdvancedIndonesianLegalPreprocessor(use_stemming=CFG.USE_STEMMING)

def process_documents(df, folder='file_putusan'):
    texts, features_list = [], []
    for doc_id in tqdm(df['id'], desc="Processing Documents"):
        file_path = Path(folder) / f"{doc_id}.txt"
        if file_path.exists():
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()
            cleaned_text = preprocessor.clean_text(text)
            features = preprocessor.extract_features(text)
            texts.append(cleaned_text)
            features_list.append(features)
        else:
            texts.append("")
            features_list.append({})
    return texts, pd.DataFrame(features_list).fillna(0)

# Define the correct path to the documents
docs_folder_path = '/kaggle/input/objective-quest-2025/file_putusan/file_putusan'

# Pass the correct path to the function
train_texts, train_features_df = process_documents(train_df, folder=docs_folder_path)
test_texts, test_features_df = process_documents(test_df, folder=docs_folder_path)

# Menyelaraskan kolom fitur dan melakukan penskalaan
# Get the union of all columns from both dataframes
all_cols = list(set(train_features_df.columns) | set(test_features_df.columns))

# Reindex both dataframes to have the same columns, filling missing values with 0
train_features_df = train_features_df.reindex(columns=all_cols).fillna(0)
test_features_df = test_features_df.reindex(columns=all_cols).fillna(0)

scaler = StandardScaler()
train_legal_features = scaler.fit_transform(train_features_df)
test_legal_features = scaler.transform(test_features_df)

print(f"Bentuk fitur rekayasa (train): {train_legal_features.shape}")


2. Prapemrosesan teks dan rekayasa fitur...


Processing Documents:   0%|          | 0/16572 [00:00<?, ?it/s]

Processing Documents:   0%|          | 0/6666 [00:00<?, ?it/s]

Bentuk fitur rekayasa (train): (16572, 32)


## 4. Pelatihan Model Transformer (Level 0)

In [ ]:
# 4. Pelatihan Model Transformer pada Seluruh Data
print("\\n4. Memulai pelatihan model Transformer pada seluruh data training...")
tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_NAME)

# Panggil fungsi untuk melatih pada seluruh data
run_full_training(train_df, tokenizer, CFG, train_texts, train_legal_features)

gc.collect()
torch.cuda.empty_cache()


4. Memulai pelatihan model Transformer (Level 0)...
Starting/Resuming training from fold 0

========== FOLD 1 ==========


Some weights of BertModel were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized because the shapes did not match:
- embeddings.position_embeddings.weight: found shape torch.Size([512, 768]) in the checkpoint and torch.Size([2048, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Using 2 GPUs!


Training:   0%|          | 0/1657 [00:00<?, ?it/s]

## 6. Inferensi Akhir & Submisi

In [ ]:
# 6. Inferensi Akhir & Submisi
print("\\n6. Menjalankan inferensi akhir pada data uji...")

# Siapkan test dataloader
test_dataset = LegalFineTuningDataset(test_df, tokenizer, CFG, test_texts, test_legal_features)
test_loader = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE * 2, shuffle=False, num_workers=4)

# Inisialisasi dan muat model yang sudah dilatih
model = LegalFineTuningModel(CFG)
model.update_feature_dim(test_legal_features.shape[1])
model.load_state_dict(torch.load("final_model.pth"))
model.to(CFG.DEVICE)
model.eval()

# Jalankan inferensi
preds = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inferencing on Test Set"):
        input_ids = batch['input_ids'].to(CFG.DEVICE)
        attention_mask = batch['attention_mask'].to(CFG.DEVICE)
        legal_features = batch['legal_features'].to(CFG.DEVICE)
        outputs = model(input_ids, attention_mask, legal_features)
        preds.append(outputs.squeeze().cpu().numpy())

final_preds_transformed = np.concatenate(preds)

# Inverse transform prediksi akhir
final_preds = final_preds_transformed.copy()
mask_final = (final_preds > 0) & (final_preds < np.log1p(88888))
final_preds[mask_final] = np.expm1(final_preds[mask_final])
final_preds = np.clip(final_preds, 0, None)
final_preds = np.round(final_preds, 2)

## 7. Buat File Submisi

In [ ]:
 # 7. Buat File Submisi
print("\n7. Membuat file submisi...")
submission = pd.DataFrame({'id': test_df['id'], 'lama hukuman (bulan)': final_preds})
submission.to_csv('submission_state_of_the_art.csv', index=False)
print("File 'submission_state_of_the_art.csv' berhasil dibuat.")

# end